In [50]:
from discovery_utils.synthesis.policy import policy_update
from discovery_utils.utils import google
import pandas as pd

In [51]:
sheet_id = "1w2nSas1LwmPQY9HK-drrxIdGpDV3wU3pePDibHPVqL8"

In [2]:
HansardData = policy_update.HansardData()

2025-02-19 17:04:56,624 - discovery_utils.getters.hansard - INFO - Downloading debates parquet file: data/policy_scanning_data/enriched/HansardDebates.parquet
2025-02-19 17:05:09,092 - discovery_utils.getters.hansard - INFO - Attempting to download label store: data/policy_scanning_data/enriched/HansardDebates_LabelStore_keywords.csv
2025-02-19 17:05:27,920 - discovery_utils.getters.hansard - INFO - Downloading people metadata
2025-02-19 17:05:29,121 - discovery_utils.getters.hansard - INFO - Successfully downloaded and saved people metadata


In [3]:
missions = ["ASF"]

In [4]:
_, data_signals = policy_update.create_policy_update_message(
    Hansard=HansardData,
    missions=missions,
    message_date="2025-02-19",
    data_start_date="2025-01-01",
    data_end_date="2025-03-31",
)

2025-02-19 17:06:39,641 - root - INFO - Summarising debate: New Homes (Solar Generation) Bill
2025-02-19 17:06:47,764 - root - INFO - Summarising debate: ECO4 and Insulation Schemes
2025-02-19 17:06:55,015 - root - INFO - Summarising debate: Climate and Nature Bill
2025-02-19 17:07:09,071 - root - INFO - Summarising debate: Energy Security and Net Zero


In [42]:
debates_cols = [
    "date",
    "title",
    "purpose",
    "positives",
    "negatives",
    "next_steps",
]

In [49]:
debates_df = (
    pd.DataFrame(data_signals[0][0]['signal'])
    .sort_values("date", ascending=True)
    .assign(
        positives = lambda df: df.positives.apply(lambda x: "\n".join(x)),
        negatives = lambda df: df.negatives.apply(lambda x: "\n".join(x)),
        next_steps = lambda df: df.next_steps.apply(lambda x: "\n".join(x)),
    )
)[debates_cols]

In [52]:
google.upload_data_to_gsheet(sheet_id, {"commons_debates": debates_df})
google.format_gsheet(sheet_id, "commons_debates", freeze_cols=2)

2025-02-19 17:23:23,533 - root - INFO - Connected to Google Sheet: Mission Radar 2025 Q1: Test data [2025-02-18]
2025-02-19 17:23:27,687 - root - INFO - Uploading DataFrame to sheet: commons_debates
/Users/karlis.kanders/Code/discovery_utils/.venv/lib/python3.11/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/karlis.kanders/Code/discovery_utils/.venv/lib/python3.11/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
2025-02-19 17:23:42,682 - root - INFO - Upload completed successfully.
2025-02-19 17:23:43,962 - root - INFO - Connected to Google Sheet: Mission Radar 2025 Q1: Test data [2025-02-18]


In [83]:
cols_highlights = [
    "date",
    "heading",
    "summary",
    "url",
    "mission_labels",
    "topic_labels",
]

In [86]:
highlights_df = []
for _debate in data_signals[0][1]['signal']:
    highlights_df.append((
        pd.DataFrame(_debate['quotes'])
        .assign(
            heading = _debate['heading'],
            date = _debate['date'],
        )
    ))
highlights_df = (
    pd.concat(highlights_df, ignore_index=True)
    .sort_values(["date", "heading"], ascending=True)
    .assign(speech_id = lambda df: df.url.apply(lambda x: "uk.org.publicwhip/debate/" + x.split("?id=")[-1]))
    .merge(
        HansardData.labelstore_df[["id", "mission_labels", "topic_labels"]],
        left_on="speech_id",
        right_on="id",
        how="left",
    )
)[cols_highlights]

In [87]:
highlights_df

,date,heading,summary,url,mission_labels,topic_labels
0,2025-01-07,Crown Estate Bill [Lords],Jones (Lab) said the Bill includes requirement...,https://www.theyworkforyou.com/debates/?id=202...,"X,ASF","Mobile,Non-tech assessment,Data science & AI,T..."
1,2025-01-07,Crown Estate Bill [Lords],Rodda (Lab) highlighted the need for significa...,https://www.theyworkforyou.com/debates/?id=202...,"X,ASF","Policy,Inclusion,Mobile,Renewables - General,D..."
2,2025-01-07,Crown Estate Bill [Lords],Heylings (Lib Dem) said the Bill aims to enhan...,https://www.theyworkforyou.com/debates/?id=202...,"X,ASF","Data science & AI,Decarbonisation - General,Re..."
3,2025-01-07,Crown Estate Bill [Lords],Onn (Lab) said that *GB Energy* and devolution...,https://www.theyworkforyou.com/debates/?id=202...,"X,ASF","Mobile,Data science & AI,Recruitment,Training,..."
4,2025-01-07,Crown Estate Bill [Lords],Murray (Lab) said the Crown Estate will co-inv...,https://www.theyworkforyou.com/debates/?id=202...,"X,ASF","Policy,Wind,Inclusion,Renewables - General,Mob..."
...,...,...,...,...,...,...
78,2025-02-04,Energy Security and Net Zero,Farron (Lib Dem) asked the Minister to meet wi...,https://www.theyworkforyou.com/debates/?id=202...,"ASF,X","Energy efficiency,Data science & AI,Decarbonis..."
79,2025-02-04,Energy Security and Net Zero,Fahnbulleh (Lab) said that there is a need to ...,https://www.theyworkforyou.com/debates/?id=202...,"ASF,X","Energy efficiency,Mobile"
80,2025-02-04,Energy Security and Net Zero,Paffey (Lab) asked what steps the Minister is ...,https://www.theyworkforyou.com/debates/?id=202...,ASF,Energy efficiency
81,2025-02-10,Biomass Generation,Law (SNP) asked when the UK Government will in...,https://www.theyworkforyou.com/debates/?id=202...,"ASF,X","Energy storage,Hydrogen energy,Data science & AI"


In [88]:
google.upload_data_to_gsheet(sheet_id, {"commons_highlights": highlights_df})
google.format_gsheet(sheet_id, "commons_highlights", freeze_cols=0)

2025-02-19 17:34:33,875 - root - INFO - Connected to Google Sheet: Mission Radar 2025 Q1: Test data [2025-02-18]
2025-02-19 17:34:38,904 - root - INFO - Uploading DataFrame to sheet: commons_highlights
/Users/karlis.kanders/Code/discovery_utils/.venv/lib/python3.11/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/karlis.kanders/Code/discovery_utils/.venv/lib/python3.11/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
2025-02-19 17:34:52,577 - root - INFO - Upload completed successfully.
2025-02-19 17:34:53,373 - root - INFO - Connected to Google Sheet: Mission Radar 2025 Q1: Test data [2025-02-18]


## Keyword counts

In [115]:
import datetime
def get_quarter_from_date(date:str) -> int:
    """Return the quarter number from a given YYYY-MM-DD date string."""
    _date = datetime.datetime.strptime(date, "%Y-%m-%d")
    return (_date.month-1)//3 + 1

In [118]:
speeches_df = (
    HansardData.debates_df
    .query("date >= '2020-01-01' & date <= '2025-03-31'")
    .merge(
        HansardData.labelstore_df[['id', 'mission_labels', 'topic_labels']],
        left_on='speech_id',
        right_on='id',
        how='left'
    )
    .assign(mission_labels = lambda df: df.mission_labels.apply(lambda x: x.split(",") if (type(x) is str) else []))
    .assign(topic_labels = lambda df: df.topic_labels.apply(lambda x: x.split(",") if (type(x) is str) else []))    
    .explode("mission_labels")
    .query("mission_labels in @missions")
    .explode("topic_labels")
    .assign(quarter = lambda df: df.date.apply(get_quarter_from_date))
    .assign(quarter = lambda df: df.year.astype(str) + "-Q" + df.quarter.astype(str))
)

In [136]:
ts_hp = (
    speeches_df
    .query("topic_labels == 'Heat pumps'")
    .groupby("quarter")
    .agg({"speech_id": "count"})
    .reset_index()
)

In [137]:
from discovery_utils.utils import charts

In [138]:
ts_hp

,quarter,speech_id
0,2020-Q1,3
1,2020-Q4,6
2,2021-Q1,4
3,2021-Q2,7
4,2021-Q3,7
5,2021-Q4,39
6,2022-Q1,16
7,2022-Q2,11
8,2022-Q3,4
9,2022-Q4,5


In [139]:
charts.ts_bar(
    ts_hp,
    variable="speech_id",
    variable_title="Number of speeches",
    time_column="quarter",

)


alt.Chart(...)

In [101]:
HansardData.debates_df.head(1)

,speech_id,speakername,speaker_id,person_id,speech,date,year,major_heading,minor_heading
0,uk.org.publicwhip/debate/2000-01-10a.1.1,NA,NA,NA,The House met at half-past Two o'clock,2000-01-10,2000,Preamble,None


In [ ]:
HansardData.labelstore_df.tail(10)

,Unnamed: 0.20,Unnamed: 0.19,Unnamed: 0.18,Unnamed: 0.17,Unnamed: 0.16,Unnamed: 0.15,Unnamed: 0.14,Unnamed: 0.13,Unnamed: 0.12,Unnamed: 0.11,...,Unnamed: 0.5,Unnamed: 0.4,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,id,mission_labels,topic_labels,text
557987,557987,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,uk.org.publicwhip/debate/2025-02-13a.468.0,"AFS,X","Play,Training,Inclusion,Creative,Mobile,Data s...","Thank you, Madam Deputy Speaker, for calling m..."
557988,557988,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,uk.org.publicwhip/debate/2025-02-13a.470.0,"AFS,X","Inclusion,Cognitive,Communication and language...",It is a huge privilege to speak in this aftern...
557989,557989,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,uk.org.publicwhip/debate/2025-02-13a.471.0,"AFS,X","Inclusion,Labour market,Cognitive,Data science...","Thank you, Madam Deputy Speaker, for allowing ..."
557990,557990,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,uk.org.publicwhip/debate/2025-02-13a.472.1,"AFS,X","Family support - General,Inclusion,Operations,...",It is a privilege to close the debate for His ...
557991,557991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,uk.org.publicwhip/debate/2025-02-13a.475.0,"AFS,X","Policy,Creative,Family support - General,Data ...",I thank all hon. Members for the constructive ...
557992,557992,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,uk.org.publicwhip/debate/2025-02-13a.475.1,X,Data science & AI,I thank the Minister for chairing an excellent...
557993,557993,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,uk.org.publicwhip/debate/2025-02-13a.475.2,X,"Inclusion,Recruitment,Mobile,Data science & AI",Absolutely. We have committed £40 million to t...
557994,557994,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,uk.org.publicwhip/debate/2025-02-13a.478.1,X,Data science & AI,"People with respiratory illnesses, such as my ..."
557995,557995,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,uk.org.publicwhip/debate/2025-02-13a.479.2,"AFS,X","Play,Family support - General,Inclusion,Traini...",It is a sad reality of life that marriages fai...
557996,557996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,uk.org.publicwhip/debate/2025-02-13a.481.0,"AFS,X","Play,Family support - General,Inclusion,Traini...",I congratulate the hon. Member for Solihull We...
